In [ ]:
# THIS WAS RUN IN COLAB; training interrupted after about 2 hours

In [1]:
! pip install mace-torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.1/453.1 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 76.6 MB/s eta 0:00:00
  Created wheel for python-hostlist: filename=python_hostlist-2.3.0-py3-none-any.whl size=39449 sha256=7a70bd524a6c43d8b811471368179f5a11e26be3441b8b1ea9cc7acab37e9d71
  Stored in directory: /root/.cache/pip/wheels/02/e4/34/75fc0cd5b7889d8cc4ce6fb2f74c9fd17b3c6138cb03832481
Successfully built python-hostlist


In [2]:
import sys
import os
from mace.cli.run_train import main

/usr/local/lib/python3.12/dist-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [3]:
# import importlib
# import useful_functions
# importlib.reload(useful_functions)
from useful_functions import clean_dataset, split_train_valid # custom functions for easier re-runs

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

In [4]:
from google.colab import files

uploaded = files.upload()

Saving OUTCAR to OUTCAR


In [5]:
import os

outcar_path = "OUTCAR"

if os.path.exists(outcar_path):
#dimension
    file_size_mb = os.path.getsize(outcar_path) / (1024 * 1024)
    print(f"Dimensione OUTCAR: {file_size_mb:.2f} MB")

#number structure
    with open(outcar_path, 'r', encoding='latin-1', errors='ignore') as f:
        n_steps = sum(1 for line in f if "POSITION" in line)

    print(f"🔄 Number structure/steps: {n_steps}")
else:
    print("File OUTCAR not found.")

Dimensione OUTCAR: 489.80 MB
🔄 Number structure/steps: 10000


In [6]:
my_full_file="./new_data/1.NVT_300/A.1_10K/cleaned_dataset.extxyz" # not yet created
my_train_file="./new_data/1.NVT_300/A.1_10K/train.extxyz" # not yet created
my_valid_file="./new_data/1.NVT_300/A.1_10K/valid.extxyz" # not yet created

my_sim_checkpoints_dir="./new_simulation/1.NVT_300/A.1_10K/checkpoints"
my_sim_results_dir="./new_simulation/1.NVT_300/A.1_10K/results"

outcar_file= "./OUTCAR"  # ./new_data/1.NVT_300/A.1_10K/OUTCAR" # unzipped (one line from terminal)

In [7]:
os.makedirs(f"{my_sim_checkpoints_dir}", exist_ok=True)
os.makedirs(f"{my_sim_results_dir}", exist_ok=True)


In [8]:
!mkdir new_data

In [9]:
!mkdir new_data/1.NVT_300

In [10]:
!mkdir new_data/1.NVT_300/A.1_10K

In [11]:
clean_dataset(
    outcar_file=outcar_file,
    destination_file=my_full_file,
    start_idx=6840 # from convergence check (visually preferred to 8400)
)

Retrieved 3160 structures from OUTCAR.
File saved as ./new_data/1.NVT_300/A.1_10K/cleaned_dataset.extxyz


0

In [12]:
split_train_valid(
    full_cleaned_extxyz_file=my_full_file,
    destination_train_file=my_train_file,
    destination_valid_file=my_valid_file,
    train_frac=0.9,
)

Total frames uploaded: 3160
Saved 2844 frames in ./new_data/1.NVT_300/A.1_10K/train.extxyz and 316 in ./new_data/1.NVT_300/A.1_10K/valid.extxyz


0

In [13]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [17]:
# hyperpars for MACE training (sources: MACE T01 tutorial; Claude AI targeted help)

my_atomic_numbers = "[1, 6, 7, 50, 53]" ##############

custom_args = [
    "mace_run_training",
    "--name=model_3000_pts",
    f"--train_file={my_train_file}",
    f"--valid_file={my_valid_file}",
    "--energy_key=energy",
    "--forces_key=forces",
    "--E0s=foundation", # try "isolated" after adding isolated atom energies to training set?
               #"mp" means from materials project (used to train foundation model)
    #"--model=MACE",
    #"--num_interactions=2", # keep default


    "--stress_key=stress",
    "--stress_weight=1.0",
    "--compute_stress=True",

    "--max_num_epochs=50",
    "--patience=10",
    "--batch_size=5",       # was 10
    "--valid_batch_size=5", # was 10
    "--device=cuda",
    "--default_dtype=float32", # less memory than 64

    "--r_max=5.0", #could be increased??


    f"--checkpoints_dir={my_sim_checkpoints_dir}",
    f"--results_dir={my_sim_results_dir}",
    "--keep_checkpoints",

    "--num_channels=64",
    "--max_L=0",
    "--seed=1",

    # use a pre-trained ("foundation") model:
    "--foundation_model=small",           # Use MACE‑MP‑0 small
    "--multiheads_finetuning=True",        # Enable recommended Multihead Replay (for generalizable models; but expensive)
    "--pt_train_file=mp",                 # Required for Multihead Replay


    f"--atomic_numbers={my_atomic_numbers}", #[1, 6, 7, 50, 53]
    "--num_samples_pt=1000",
    "--lr=0.0001",

    "--energy_weight=1.0", ##############
    "--forces_weight=100.0", #############

#OPTIMIZER
    "--optimizer=adamw",  #AdamW - optimizer
    "--amsgrad", #AMSGrad variant --> ensure stability
    "--weight_decay=5e-7",


#EMA/SWA
#optimization technique and loss function
#EMA: exponential smoothing
#SWA: stochastic weight averaging

#ema: Enables Exponential Moving Average during training.
#Instead of saving only the model weights from the current optimization step, it maintains a running average of the model parameters.
#This produces a smoother, more stable set of parameters, reducing variance and helping the model generalize better on unseen structures.
    "--ema",
    "--ema_decay=0.99",  #decay rate

#swa: Enables Stochastic Weight Averaging, a technique that averages model weights sampled across multiple late stage epochs
#find a broader, flatter local minimum in the loss landscape.
    "--swa",
    "--start_swa=30", #averaging will begin at epoch start_swa

#importance
    "--swa_forces_weight=100.0",
    "--swa_stress_weight=1.0",
    "--swa_energy_weight=1.0",


#Dynamically normalizes the loss contributions based on the Root Mean Square (RMS) of the atomic forces in the dataset.
#This prevents large force magnitudes from dominating the gradients unevenly across different atomic configurations.
    "--scaling=rms_forces_scaling",


]

sys.argv = custom_args

In [18]:
if __name__ == "__main__":
    try:
        main()
        print("Training completed!")
    except Exception as e:
        print(f"Error: {e}")

INFO:root:===========VERIFYING SETTINGS===========


2026-08-14 11:13:57.897 INFO: ===========VERIFYING SETTINGS===========


INFO:root:Stage Two is activated as start_stage_two was defined


2026-08-14 11:13:57.899 INFO: Stage Two is activated as start_stage_two was defined


INFO:root:MACE version: 0.3.16


...

2026-08-14 12:12:09.522 INFO: Epoch 18: head: pt_head, loss=0.01398842, RMSE_E_per_atom=   67.45 meV, RMSE_F=  141.84 meV / A, RMSE_stress=    7.48 meV / A^3


INFO:root:Epoch 18: head: Default, loss=0.00474102, RMSE_E_per_atom=    0.82 meV, RMSE_F=   39.93 meV / A, RMSE_stress=    5.69 meV / A^3


2026-08-14 12:12:15.959 INFO: Epoch 18: head: Default, loss=0.00474102, RMSE_E_per_atom=    0.82 meV, RMSE_F=   39.93 meV / A, RMSE_stress=    5.69 meV / A^3


DEBUG:root:Saving checkpoint: ./new_simulation/1.NVT_300/A.1_10K/checkpoints/model_3000_pts_run-1_epoch-18.pt


KeyboardInterrupt: 

In [19]:
files.download("/content/new_simulation/1.NVT_300/A.1_10K/checkpoints/model_3000_pts_run-1_epoch-17.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
files.download("/content/model_3000_pts_valid_indices_1.txt")
files.download("/content/mp_finetuning-model_3000_pts_run-1_combined.xyz")

# others can be downloaded manually

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# RMSE_F (forces): Default head - for MD-quality models, sub-50 meV/Å is commonly considered good, sub-20-30 meV/Å is very good.
# RMSE_E_per_atom (energy): Default head's sub-few-meV/atom is typical for usable models.

In [ ]:
# TO DO: a) fix warnings (expecially "energy" and "forces" keys),
# b) test??